# 🚀 t4-diffusion: Fast Stable Diffusion on Free Colab T4

This notebook demonstrates the complete workflow for running optimized Stable Diffusion on Google Colab's free T4 GPU.

**Features:**
- INT8 quantization with TensorRT for up to 2x speedup
- Feature caching for inference acceleration
- VRAM monitoring with 15.6GB T4 limit enforcement
- Engine save/load for faster subsequent runs

**Requirements:**
- Google Colab with T4 GPU runtime
- ~10GB free VRAM

## 1. Setup & Installation

In [ ]:
# Install t4-diffusion from GitHub
!pip install git+https://github.com/Kash6/t4-diffusion.git -q
!pip install hypothesis pytest -q  # For running tests

In [ ]:
# Verify GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {gpu_name}")
    print(f"✓ VRAM: {total_vram:.1f} GB")
    
    if "T4" in gpu_name:
        print("✓ Running on T4 GPU (optimal for this package)")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime → Change runtime type → T4 GPU")

## 2. Basic Inference with SDXL-Turbo

In [ ]:
from diffusion_trt.pipeline import PipelineConfig, OptimizedPipeline
from diffusion_trt.utils.vram_monitor import VRAMMonitor

# Configure the pipeline
config = PipelineConfig(
    model_id="stabilityai/sdxl-turbo",
    enable_int8=True,           # Enable INT8 quantization
    enable_caching=True,        # Enable feature caching
    num_inference_steps=4,      # SDXL-Turbo uses 4 steps
    guidance_scale=0.0,         # No CFG for SDXL-Turbo
    seed=42,                    # For reproducible results
)

print("Configuration:")
print(f"  Model: {config.model_id}")
print(f"  INT8: {config.enable_int8}")
print(f"  Caching: {config.enable_caching}")
print(f"  Steps: {config.num_inference_steps}")

In [ ]:
# Load and optimize the model
# This takes 2-5 minutes on first run (downloading + compiling)
print("Loading and optimizing model...")
print("This may take a few minutes on first run.")

with VRAMMonitor() as monitor:
    pipeline = OptimizedPipeline.from_pretrained(
        config.model_id,
        config=config,
    )

print(f"\n✓ Model loaded!")
print(f"✓ Peak VRAM: {monitor.peak_gb:.2f} GB")

In [ ]:
# Generate an image
prompt = "A beautiful sunset over a calm ocean, vibrant colors, photorealistic"

print(f"Generating: '{prompt}'")

with VRAMMonitor() as monitor:
    images = pipeline(prompt)

print(f"\n✓ Generated!")
print(f"✓ VRAM used: {monitor.peak_gb:.2f} GB")

# Display the image
images[0]

## 3. VRAM Monitoring

In [ ]:
from diffusion_trt.utils.vram_monitor import get_vram_usage, get_peak_vram, clear_cache
from diffusion_trt.models import T4_VRAM_LIMIT_GB

print(f"T4 VRAM Limit: {T4_VRAM_LIMIT_GB} GB")
print(f"Current VRAM: {get_vram_usage():.2f} GB")
print(f"Peak VRAM: {get_peak_vram():.2f} GB")
print(f"Available: {T4_VRAM_LIMIT_GB - get_vram_usage():.2f} GB")

In [ ]:
# Monitor VRAM during multiple generations
prompts = [
    "A majestic mountain landscape with snow-capped peaks",
    "A futuristic cityscape at night with neon lights",
    "A cozy cabin in the woods during autumn",
]

for i, prompt in enumerate(prompts, 1):
    with VRAMMonitor() as monitor:
        images = pipeline(prompt)
    
    print(f"Image {i}: VRAM={monitor.peak_gb:.2f}GB | '{prompt[:40]}...'")
    display(images[0])

## 4. Feature Caching Demonstration

In [ ]:
# Check cache statistics
cache_stats = pipeline.get_cache_stats()

if cache_stats:
    print("Cache Statistics:")
    print(f"  Hits: {cache_stats.get('hits', 0)}")
    print(f"  Misses: {cache_stats.get('misses', 0)}")
    print(f"  Hit Rate: {cache_stats.get('hit_rate', 0):.1%}")
    print(f"  Cache Size: {cache_stats.get('total_size_bytes', 0) / 1e6:.1f} MB")
else:
    print("Caching not enabled or no stats available")

In [ ]:
# Clear cache to free memory
pipeline.clear_cache()
print(f"Cache cleared. Current VRAM: {get_vram_usage():.2f} GB")

## 5. Engine Save/Load for Faster Startup

In [ ]:
# Save the optimized engine
engine_path = "optimized_sdxl_turbo.pt"
pipeline.save_engine(engine_path)
print(f"✓ Engine saved to: {engine_path}")

In [ ]:
# Free memory before loading the saved engine
# (T4 can't hold two pipelines simultaneously)
print(f"VRAM before cleanup: {get_vram_usage():.2f} GB")

del pipeline
import gc
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after cleanup: {get_vram_usage():.2f} GB")

In [ ]:
# Load the engine (much faster than re-optimizing)
import time

start = time.time()
loaded_pipeline = OptimizedPipeline.load_engine(engine_path)
load_time = time.time() - start

print(f"✓ Engine loaded in {load_time:.1f}s")
print(f"  (vs 2-5 minutes for full optimization)")

In [ ]:
# Verify loaded engine produces same output
test_prompt = "A serene Japanese garden with cherry blossoms"

# Generate with loaded pipeline
loaded_images = loaded_pipeline(test_prompt, seed=42)
print("Generated with loaded engine:")
display(loaded_images[0])

# Use loaded_pipeline for remaining cells
pipeline = loaded_pipeline

## 6. Benchmark Comparison

In [ ]:
# Run benchmark
print("Running benchmark (5 iterations + 2 warmup)...")

metrics = pipeline.benchmark(
    prompt="A beautiful landscape with mountains and a lake",
    num_iterations=5,
    warmup_iterations=2,
)

print("\n" + "="*50)
print("BENCHMARK RESULTS")
print("="*50)
print(f"Latency (mean):  {metrics.latency_mean_ms:.0f} ms")
print(f"Latency (std):   {metrics.latency_std_ms:.0f} ms")
print(f"Latency (p50):   {metrics.latency_p50_ms:.0f} ms")
print(f"Latency (p95):   {metrics.latency_p95_ms:.0f} ms")
print(f"Throughput:      {metrics.throughput_images_per_sec:.2f} img/s")
print(f"Peak VRAM:       {metrics.vram_peak_gb:.2f} GB")
print(f"Cache Hit Rate:  {metrics.cache_hit_rate:.1%}")

## 7. Batch Inference

In [ ]:
# Generate multiple images with different prompts
batch_prompts = [
    "A cyberpunk street scene with rain and neon signs",
    "An ancient temple hidden in a misty forest",
    "A steampunk airship flying over Victorian London",
    "A magical library with floating books and glowing orbs",
]

print(f"Generating {len(batch_prompts)} images...")

import time
start = time.time()

batch_images = []
for prompt in batch_prompts:
    images = pipeline(prompt)
    batch_images.extend(images)

total_time = time.time() - start
avg_time = total_time / len(batch_prompts)

print(f"\n✓ Generated {len(batch_images)} images")
print(f"✓ Total time: {total_time:.1f}s")
print(f"✓ Average: {avg_time:.1f}s per image")
print(f"✓ Throughput: {len(batch_images)/total_time:.2f} img/s")

In [ ]:
# Display batch results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, img, prompt in zip(axes.flat, batch_images, batch_prompts):
    ax.imshow(img)
    ax.set_title(prompt[:40] + "...", fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Run Tests (Optional)

In [ ]:
# Run unit tests
!python -m pytest tests/unit -v --tb=short 2>&1 | tail -30

In [ ]:
# Run property tests
!python -m pytest tests/property -v --tb=short 2>&1 | tail -30

## 9. Cleanup

In [ ]:
# Clear all caches and free memory
pipeline.clear_cache()
clear_cache()

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"Final VRAM usage: {get_vram_usage():.2f} GB")

---

## Summary

This notebook demonstrated:

1. **Installation** - Simple pip install from GitHub
2. **Basic Inference** - Generate images with SDXL-Turbo
3. **VRAM Monitoring** - Track memory usage to stay within T4 limits
4. **Feature Caching** - Accelerate inference with cached features
5. **Engine Persistence** - Save/load optimized engines for faster startup
6. **Benchmarking** - Measure latency and throughput
7. **Batch Inference** - Generate multiple images efficiently
8. **Testing** - Run the comprehensive test suite

For more information, visit: https://github.com/Kash6/t4-diffusion